In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input


In [2]:
# Load the CSV file
df = pd.read_csv("Final_SCATS_with_LatLong.csv")


In [5]:
# Clean up and sort data
df.dropna(inplace=True)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values(by='Timestamp')


In [7]:
# Training on the busiest site
site_id = df['Site ID'].value_counts().idxmax()
train_data = df[df['Site ID'] == site_id]


In [9]:
train_data = train_data.set_index('Timestamp').resample('h').sum()
scaler = MinMaxScaler()
scaled = scaler.fit_transform(train_data[['Volume']])


In [11]:
# Createing sequence and prediction
def create_sequences(data, window=4):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled, window=4)
X = X.reshape((X.shape[0], X.shape[1], 1))


In [ ]:
# LSTM model
model = Sequential()
model.add(Input(shape=(X.shape[1], 1)))
model.add(LSTM(64))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
